In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
 
from sklearn.linear_model import Ridge
 
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

from xgboost import XGBRegressor

In [ ]:
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',1000)

In [ ]:
train=pd.read_csv('./X_train.csv', index_col=0)
test=pd.read_csv('./X_test.csv', index_col=0)
price_res=pd.read_csv('./y_train.csv', index_col=0)
print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
print(f"price shape: {price_res.shape}")

# pipeline

In [314]:
import pandas as pd
from datetime import datetime
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, LabelEncoder,OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns)

class DateTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['建築完成年月'] = pd.to_datetime(df['建築完成年月'])
        today = datetime.today()
        df['date_str'] = df['交易年'].astype(str) + '-' + df['交易月'].astype(str).str.zfill(2) + '-' + df['交易日'].astype(str).str.zfill(2)
        df['交易日期'] = pd.to_datetime(df['date_str'])
        df['建築年紀'] = df['交易日期'].dt.year - df['建築完成年月'].dt.year
        df = df.drop(['建築完成年月', 'date_str','交易日期'], axis=1)
        
        return df
class FloorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['所在樓層'] = df['移轉層次']/df['總樓層數']
       
         
        
        return df

class AddShoppingPlaceFeature(BaseEstimator, TransformerMixin): 
    def fit(self, X, y=None): 
        return self 
    def transform(self, X): 
        X = X.copy() 
        X['購物場所'] = X[['大賣場', '超市', '百貨公司']].mean(axis=1) 
        return X 
class AddSchoolFeature(BaseEstimator, TransformerMixin): 
        def fit(self, X, y=None):
            return self 
        def transform(self, X): 
            X = X.copy() 
            X['學校'] = X[['托兒所', '國中', '高中職', '大學']].mean(axis=1) 
            return X
class AddGovernmentFeature(BaseEstimator, TransformerMixin): 
        def fit(self, X, y=None):
            return self 
        def transform(self, X): 
            X = X.copy() 
            X['政府'] = X[['警察局', '消防局']].mean(axis=1) 
            return X

class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.label_encoders = {}

    def fit(self, X, y=None):
        for col in X.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.label_encoders[col] = le
        return self

    def transform(self, X):
        df = X.copy()
        for col, le in self.label_encoders.items():
            df[col] = le.transform(df[col])
        return df

class OneHotEncoderTransformer(BaseEstimator,  
 TransformerMixin):
    def __init__(self, handle_unknown='ignore'):
        """
        Initializes the OneHotEncoderTransformer.

        Args:
            handle_unknown (str, default='ignore'): Specifies how to handle
                unknown categories during transformation. Possible values are:
                - 'error': Raise an error for unknown categories.
                - 'ignore': Ignore unknown categories and treat them as
                  previously unseen categories.
        """
        self.encoders = {}
        self.handle_unknown = handle_unknown

    def fit(self, X, y=None):
        """
        Fits the transformer to the data X.

        Args:
            X (pd.DataFrame): The data to fit the transformer on.
            y (None, optional): Not used in this context.

        Returns:
            OneHotEncoderTransformer: The fitted transformer.
        """
        self.numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
        for col in X.select_dtypes(include=['object']).columns:
            #if col not in self.numerical_cols:
                encoder = OneHotEncoder(handle_unknown=self.handle_unknown)
                encoder.fit(X[[col]])  # Reshape to 2D for OneHotEncoder
                self.encoders[col] = encoder
        return self

    def transform(self, X):
        """
        Transforms the data X using one-hot encoding.

        Args:
            X (pd.DataFrame): The data to transform.

        Returns:
            pd.DataFrame: The transformed data with one-hot encoded columns 
                         and original numerical columns.
        """
        encoded_df = X[self.numerical_cols].copy()
        for col, encoder in self.encoders.items():
            encoded_features = encoder.transform(X[[col]]).toarray()
            encoded_df = pd.concat([encoded_df, pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out([col]))], axis=1)
        return encoded_df
class GaussianBasisFunctionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, ngrid=3):
        self.ngrid = ngrid

    def fit(self, X, y=None):
        coord_x = X['橫坐標'].values
        coord_y = X['縱坐標'].values

        xmin, xmax = min(coord_x), max(coord_x) 
        ymin, ymax = min(coord_y), max(coord_y) 
        self.xgrids = np.linspace(xmin, xmax, self.ngrid + 1)
        self.ygrids = np.linspace(ymin, ymax, self.ngrid + 1)

        self.mu_x_all, self.mu_y_all, self.std_x_all, self.std_y_all, self.count_all = [], [], [], [], []

        for i in range(self.ngrid):
            for j in range(self.ngrid):
                x1, x2 = self.xgrids[i], self.xgrids[i + 1]
                y1, y2 = self.ygrids[j], self.ygrids[j + 1]
                tmpindx = (x1 <= coord_x) * (coord_x < x2)
                tmpindy = (y1 <= coord_y) * (coord_y < y2)
                tmpind = tmpindx * tmpindy
                npoints = np.sum(tmpind)

                if npoints >= 20:
                    mu_x = np.mean(coord_x[tmpind])
                    mu_y = np.mean(coord_y[tmpind])
                    std_x = np.std(coord_x[tmpind])
                    std_y = np.std(coord_y[tmpind])

                    self.mu_x_all.append(mu_x)
                    self.mu_y_all.append(mu_y)
                    self.std_x_all.append(std_x)
                    self.std_y_all.append(std_y)
                    self.count_all.append(npoints)

        return self

    def transform(self, X):
        coord_x = X['橫坐標'].values
        coord_y = X['縱坐標'].values

        ngf = len(self.mu_x_all)
        gf_all = np.zeros((coord_x.shape[0], ngf))

        for ii in range(ngf):
            mu_x = self.mu_x_all[ii]
            mu_y = self.mu_y_all[ii]
            std_x = self.std_x_all[ii]
            std_y = self.std_y_all[ii]

            tmpgf = np.exp(-(coord_x - mu_x) ** 2 / (2 * std_x ** 2) - (coord_y - mu_y) ** 2 / (2 * std_y ** 2))
            gf_all[:, ii] = tmpgf

        gf_df = pd.DataFrame(gf_all, columns=[f"gf_{i}" for i in range(gf_all.shape[1])])
        
        return pd.concat([X, gf_df], axis=1)

# Create the pipeline
pipeline = Pipeline([
    ('date_transformer', DateTransformer()),
    ('floor_transformer',FloorTransformer()),
   #('add_shopping_place', AddShoppingPlaceFeature()), 
  # ('add_school', AddSchoolFeature()),
  #('add_GovernmentFeature', AddGovernmentFeature()),
 ('drop_unwanted_columns', DropColumns(columns=['托兒所','國小', '國中', '高中職', '大學', '大賣場', '超市', '百貨公司'])),
# ('label_encoder', LabelEncoderTransformer()),
 #('onehot encoder',OneHotEncoderTransformer()),
   ('gaussian_basis', GaussianBasisFunctionTransformer(ngrid=3)),
  


])




In [338]:
full=train.copy()
full_transformed=pipeline.fit_transform(full)
# Apply the pipeline to your data
y_test = test.copy()
test_transformed = pipeline.fit_transform(y_test)


In [339]:
# 填補類別型特徵的缺失值
categorical_cols = full_transformed.select_dtypes(include=['object']).columns

# 對所有類別型特徵進行獨熱編碼
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
full_encoded = onehot_encoder.fit_transform(full_transformed[categorical_cols])
test_encoded = onehot_encoder.transform(test_transformed[categorical_cols])
# 將編碼後的結果轉為 DataFrame 並確保列名稱為字符串
full_encoded_df = pd.DataFrame(full_encoded, columns=onehot_encoder.get_feature_names_out(categorical_cols).astype(str))
test_encoded_df = pd.DataFrame(test_encoded, columns=onehot_encoder.get_feature_names_out(categorical_cols).astype(str))
print(test_encoded_df.shape,full_encoded_df.shape)
# 刪除原始類別特徵，並添加編碼後的數據
full_transformed = full_transformed.drop(categorical_cols, axis=1).reset_index(drop=True)
test_transformed = test_transformed.drop(categorical_cols, axis=1).reset_index(drop=True)

full_transformed = pd.concat([full_transformed, full_encoded_df], axis=1)
test_transformed = pd.concat([test_transformed, test_encoded_df], axis=1)

(2366, 692) (9460, 692)


In [341]:
full_transformed.shape,test_transformed.shape

((9460, 724), (2366, 724))

# feature select

## corr 

In [223]:
import pandas as pd

def select_features(df1=price_res, df2=full_transformed
                    , threshold=0.01):
    """
    Selects features from df2 based on correlation with a target column in df1.

    Args:
        df1: DataFrame containing the target column.
        df2: DataFrame containing potential features.
        threshold: Correlation threshold for feature selection.

    Returns:
        DataFrame: Filtered DataFrame with selected features.
    """

    train_corr = df2.corrwith(df1['單價元平方公尺'], axis=0)
    train_corr = abs(train_corr)

    selected_features = train_corr[train_corr > threshold].index
    
    return selected_features

In [224]:
selected_fetures=select_features()
print(selected_fetures.shape)
full_selected=full_transformed[selected_fetures]

(516,)


In [225]:
 
missing_cols = set(full_transformed.columns) - set(test_transformed.columns)
for col in missing_cols:
   test_transformed[col] = 0
test_selected=test_transformed[selected_fetures]


In [226]:
len(full_selected.columns),len(test_selected.columns)

(516, 516)

# model & Evaluate

In [227]:
from sklearn .model_selection import KFold

## score

In [228]:
# define cross validation strategy
def rmse_cv(model,X,y):
    
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=kfold))
    return rmse


In [210]:
class grid():
    def __init__(self,model):
        self.model = model
    
    def grid_get(self,X,y,param_grid):
        grid_search = GridSearchCV(self.model,param_grid,cv=5, scoring="neg_mean_squared_error")
        grid_search.fit(X,y)
        print(grid_search.best_params_, np.sqrt(-grid_search.best_score_))
        grid_search.cv_results_['mean_test_score'] = np.sqrt(-grid_search.cv_results_['mean_test_score'])
        print(pd.DataFrame(grid_search.cv_results_)[['params','mean_test_score','std_test_score']])

### parameter
- xgb
  - 'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 1.0
- rf
  - 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20
- extra
  - 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300

In [270]:
param_grid = { 'learning_rate': [0.01, 0.05, 0.1],
              'max_depth': [4, 6, 8],
              'min_child_weight': [1, 3, 5], 
              'subsample': [0.8, 1.0],
              'colsample_bytree': [0.6, 0.8, 1.0], 
              'n_estimators': [100, 500, 1000],
              'tree_method': ['gpu_hist'] }

In [271]:
# Initialize the XGBRegressor with GPU support 
xgb = XGBRegressor()
# Initialize the grid class with the XGBRegressor model 
grid_search = grid(xgb) # Assuming X_train and y_train are your training data
grid_search.grid_get(full_transformed,price_res['單價元平方公尺'], param_grid)

{'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 5, 'n_estimators': 1000, 'subsample': 0.8, 'tree_method': 'gpu_hist'} 32613.062198930016
                                                params  mean_test_score  \
0    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     44712.541746   
1    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     44711.198494   
2    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     37182.450526   
3    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     37256.239016   
4    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     35789.805675   
5    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     36072.249526   
6    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     44712.276484   
7    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     44712.437507   
8    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     37170.611135   
9    {'colsample_bytree': 0.6, 'learning_rate': 0.0...     37261.375175   
1

In [265]:
xgb=XGBRegressor( learning_rate= 0., max_depth= 8, min_child_weight= 1, subsample= 1.0, n_estimators=2000, tree_method='gpu_hist')
rf=RandomForestRegressor(max_depth= None, max_features= 'sqrt', min_samples_leaf= 1, min_samples_split= 2, n_estimators= 20,n_jobs=-1)
extra=ExtraTreesRegressor(max_depth= None, min_samples_leaf= 2, min_samples_split= 2, n_estimators= 300,n_jobs=-1)
rg = Ridge(alpha=0.1, random_state=1001)
rf2 = RandomForestRegressor(n_estimators=300, min_samples_split=3, random_state=1001, n_jobs=-1)
gbr = GradientBoostingRegressor()

### parameters 
'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 5, 'n_estimators': 1000, 'subsample': 0.8, 'tree_method': 'gpu_hist'} 32613.062198930016

# stacking


In [266]:
from sklearn.ensemble import StackingRegressor 

In [268]:
stack = StackingRegressor(
    estimators=[
        #('extra', extra),
        ('rf', rf2),
        ('xgb',xgb),
      
    ],
    final_estimator=rg
)

In [ ]:
score =rmse_cv(xgb,full_transformed,price_res['單價元平方公尺'])#,  rmse_cv(xgb,full_selected,price_res['單價元平方公尺']).mean()
score.mean()

In [ ]:
score.mean()

In [ ]:
stack.fit(full_transformed,price_res['單價元平方公尺'])

In [ ]:
res=stack.predict(test_selected)


 
# 建立 DataFrame
df = pd.DataFrame({'單價元平方公尺': res})

# 新增 Id 欄位
# Create a DataFrame, prioritizing 'Id'
df = pd.DataFrame({'Id': df.index, '單價元平方公尺': res})

# Specify the output directory
output_dir = './output'  # Replace with your desired path
output_file = output_dir + '/output.csv'



# # 將 DataFrame 輸出為 CSV 檔案
df.to_csv(output_file, index=False)

# gpu support

In [301]:
def compute_feature_importances(rf_model):
    total_importances = np.zeros(rf_model.n_features_) 
    for tree in rf_model._forest: 
        tree_importances = tree.feature_importances_ 
        total_importances += tree_importances 
        return total_importances / len(rf_model._forest)

In [317]:
import cudf

from cuml.linear_model import Ridge
from cuml.ensemble import RandomForestRegressor 
#from cuml.ensemble import StackingRegressor
#from cuml.experimental.ensemble import ExtraTreesRegressor
#from cuml.preprocessing.model_selection import cross_val_score
from cuml.ensemble import RandomForestRegressor

from cuml.linear_model import Ridge
from xgboost import XGBRegressor

# Create the models with GPU support
xgb = XGBRegressor(learning_rate=0.05,max_depth=8, min_child_weight=5, subsample=0.8,n_estimators=1000, tree_method='gpu_hist')
rf = RandomForestRegressor(max_depth=10, max_features='sqrt', min_samples_leaf=1, min_samples_split=2, n_estimators=20,random_state=1001  )
#extra = ExtraTreesRegressor(max_depth=None, min_samples_leaf=2, min_samples_split=2, n_estimators=300,  )
rg = Ridge(alpha=0.1, random_state=1001)
rf2 = RandomForestRegressor(n_estimators=300, min_samples_split=3, random_state=1001, )
 
# Assuming your data is in cuDF DataFrame
#score = rmse_cv(  full_transformed, price_res['單價元平方公尺'])
#print("Mean RMSE:", score.mean())
# Assuming rmse_cv is a custom function for cross-validation scoring
# You can use cuML's cross_val_score for this



[I] [00:45:53.225888] Unused keyword parameter: random_state during cuML estimator initialization


In [318]:
class AverageWeight(BaseEstimator, RegressorMixin):
    def __init__(self,mod,weight):
        self.mod = mod
        self.weight = weight
        
    def fit(self,X,y):
        self.models_ = [clone(x) for x in self.mod]
        for model in self.models_:
            model.fit(X,y)
        return self
    
    def predict(self,X):
        w = list()
        pred = np.array([model.predict(X) for model in self.models_])
        # for every data point, single model prediction times weight, then add them together
        for data in range(pred.shape[1]):
            single = [pred[model,data]*weight for model,weight in zip(range(pred.shape[0]),self.weight)]
            w.append(np.sum(single))
        return w

In [319]:
weight_avg = AverageWeight(mod = [xgb,rf],weight=[0.5,0.5])

In [320]:
 
score =rmse_cv(weight_avg,full_transformed,price_res['單價元平方公尺'])
score.mean()

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/conda/envs/myenv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/tmp/ipykernel_5315/1218309604.py", line 9, in fit
    model.fit(X,y)
  File "/opt/conda/envs/myenv/lib/python3.10/site-packages/xgboost/core.py", line 726, in inner_f
    return func(**kwargs)
  File "/opt/conda/envs/myenv/lib/python3.10/site-packages/xgboost/sklearn.py", line 1081, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
  File "/opt/conda/envs/myenv/lib/python3.10/site-packages/xgboost/sklearn.py", line 596, in _wrap_evaluation_matrice